In [1]:
import pandas as pd 
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error , mean_squared_error
import joblib
import warnings
warnings.filterwarnings('ignore')

In [2]:
train_df = pd.read_parquet("../data/processed/train_corrected.parquet")
val_df   = pd.read_parquet("../data/processed/val_split.parquet")

print(f"Train: {train_df.shape}")
print(f"Val:   {val_df.shape}")

Train: (2200000, 37)
Val:   (450000, 36)


In [3]:
# Features used for training
feature_cols = [
    'store_id', 'product_id', 'city_id',
    'first_category_id', 'second_category_id', 'third_category_id',
    'day_of_week', 'week_of_year', 'month', 'is_weekend',
    'sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_lag_28',
    'rolling_mean_7', 'rolling_std_7', 'rolling_mean_14', 'rolling_std_14',
    'rolling_mean_28', 'rolling_std_28', 'rolling_max_7', 'rolling_min_7',
    'stockout_lag_1', 'stockout_count_7', 'stockout_count_28',
    'discount', 'holiday_flag', 'activity_flag',
    'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level'
]

In [ ]:
# Two targets - naive vs corrected
target_naive     = 'sale_amount'
target_corrected = 'demand_corrected'


In [5]:
X_train = train_df[feature_cols]
y_train_naive = train_df[target_naive]

X_val = val_df[feature_cols]
y_val = val_df[target_naive]


In [6]:
# Naive model trains on raw observed sales
model_naive = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

In [7]:
model_naive.fit(X_train, y_train_naive)
print("Naive model trained")

Naive model trained


In [8]:
y_train_corrected = train_df[target_corrected]

In [9]:
model_corrected = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model_corrected.fit(X_train, y_train_corrected)
print("Corrected model trained")

Corrected model trained


In [10]:
pred_naive     = model_naive.predict(X_val)
pred_corrected = model_corrected.predict(X_val)

In [11]:
def evaluate(y_true, y_pred, name):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    wape = np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))
    print(f"\n{name}")
    print(f"  MAE:  {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  WAPE: {wape:.4f}")

In [12]:
evaluate(y_val, pred_naive,     "Naive Model")
evaluate(y_val, pred_corrected, "Corrected Model")


Naive Model
  MAE:  0.3472
  RMSE: 0.5744
  WAPE: 0.3141

Corrected Model
  MAE:  0.3507
  RMSE: 0.5663
  WAPE: 0.3173


In [13]:
# Evaluate only on stockout days in validation set
stockout_mask = val_df['stockout_flag'] == 1

y_val_stk = y_val[stockout_mask]
pred_naive_stk     = pred_naive[stockout_mask]
pred_corrected_stk = pred_corrected[stockout_mask]

In [14]:
evaluate(y_val_stk, pred_naive_stk,     "Naive Model (stockout days only)")
evaluate(y_val_stk, pred_corrected_stk, "Corrected Model (stockout days only)")


Naive Model (stockout days only)
  MAE:  0.3787
  RMSE: 0.6286
  WAPE: 0.3364

Corrected Model (stockout days only)
  MAE:  0.3920
  RMSE: 0.6495
  WAPE: 0.3483


In [15]:
print("Average predictions on stockout days:")
print(f"  Naive model:     {pred_naive_stk.mean():.4f}")
print(f"  Corrected model: {pred_corrected_stk.mean():.4f}")
print(f"  Actual sales:    {y_val_stk.mean():.4f}")

print("\nAverage predictions on normal days:")
normal_mask = val_df['stockout_flag'] == 0
print(f"  Naive model:     {pred_naive[normal_mask].mean():.4f}")
print(f"  Corrected model: {pred_corrected[normal_mask].mean():.4f}")
print(f"  Actual sales:    {y_val[normal_mask].mean():.4f}")

Average predictions on stockout days:
  Naive model:     1.1242
  Corrected model: 1.2036
  Actual sales:    1.1255

Average predictions on normal days:
  Naive model:     1.0592
  Corrected model: 1.1173
  Actual sales:    1.0935


In [16]:
import os
os.makedirs("../artifacts/models", exist_ok=True)

In [17]:
joblib.dump(model_naive,     "../artifacts/models/model_naive.pkl")
joblib.dump(model_corrected, "../artifacts/models/model_corrected.pkl")
joblib.dump(feature_cols,    "../artifacts/models/feature_cols.pkl")

print("Saved:")
print("  artifacts/models/model_naive.pkl")
print("  artifacts/models/model_corrected.pkl")
print("  artifacts/models/feature_cols.pkl")

Saved:
  artifacts/models/model_naive.pkl
  artifacts/models/model_corrected.pkl
  artifacts/models/feature_cols.pkl
